# Building a Movie Recommendation System with Collaborative Filtering

This notebook is a practical implementation of the tutorial found on freeCodeCamp: [How to Build a Movie Recommendation System Based on Collaborative Filtering](https://www.freecodecamp.org/news/how-to-build-a-movie-recommendation-system-based-on-collaborative-filtering/).

We will implement a movie recommendation system using **item-based collaborative filtering**. The core idea is to find movies that are similar to each other based on user ratings. We'll use Singular Value Decomposition (SVD) to discover latent features and Cosine Similarity to measure the likeness between movies.

### Prerequisites
1.  **Python Libraries**: `pandas` and `scikit-learn`.
2.  **Dataset**: The MovieLens "latest-small" dataset. You must download `movies.csv` and `ratings.csv` and place them in the same directory as this notebook.

## Step 0: Import Libraries

In [1]:
import pandas as pd
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity

## Step 1: Data Loading and Preprocessing

In [2]:
# Load the datasets
movies_df = pd.read_csv('movies.csv')
ratings_df = pd.read_csv('ratings.csv')

In [3]:
# Display the first few rows of the movies dataframe
print("Movies DataFrame:")
movies_df.head()

Movies DataFrame:


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [4]:
# Display the first few rows of the ratings dataframe
print("Ratings DataFrame:")
ratings_df.head()

Ratings DataFrame:


,userId,movieId,rating,timestamp
0,1,296,5.0,1147880044
1,1,306,3.5,1147868817
2,1,307,5.0,1147868828
3,1,665,5.0,1147878820
4,1,899,3.5,1147868510


### Merge the DataFrames
We merge the two dataframes on `movieId` to have the movie titles and ratings in a single place.

In [5]:
df = pd.merge(ratings_df, movies_df, on='movieId')

In [6]:
# Display the first few rows of the merged dataframe
df.head()

,userId,movieId,rating,timestamp,title,genres
0,1,296,5.0,1147880044,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller
1,1,306,3.5,1147868817,Three Colors: Red (Trois couleurs: Rouge) (1994),Drama
2,1,307,5.0,1147868828,Three Colors: Blue (Trois couleurs: Bleu) (1993),Drama
3,1,665,5.0,1147878820,Underground (1995),Comedy|Drama|War
4,1,899,3.5,1147868510,Singin' in the Rain (1952),Comedy|Musical|Romance


In [7]:
# Get a quick statistical summary
df.describe()

,userId,movieId,rating,timestamp
count,2.500010e+07,2.500010e+07,2.500010e+07,2.500010e+07
mean,8.118928e+04,2.138798e+04,3.533854e+00,1.215601e+09
std,4.679172e+04,3.919886e+04,1.060744e+00,2.268758e+08
min,1.000000e+00,1.000000e+00,5.000000e-01,7.896520e+08
25%,4.051000e+04,1.196000e+03,3.000000e+00,1.011747e+09
50%,8.091400e+04,2.947000e+03,3.500000e+00,1.198868e+09
75%,1.215570e+05,8.623000e+03,4.000000e+00,1.447205e+09
max,1.625410e+05,2.091710e+05,5.000000e+00,1.574328e+09


In [8]:
# Check for missing values
df.isnull().sum()

userId       0
movieId      0
rating       0
timestamp    0
title        0
genres       0
dtype: int64

The dataset is clean with no missing values. Now, we'll filter the data to improve the quality and performance of our model.

### Filtering the Data
To reduce computational complexity and noise, we will:
1.  Remove users who have rated fewer than 50 movies.
2.  Remove movies that have been rated by fewer than 50 users.

In [9]:
# Count how many ratings each user has given
user_rating_counts = df['userId'].value_counts()

# Identify users who have rated 50 or more movies
active_users = user_rating_counts[user_rating_counts >= 50].index

# Filter the dataframe to keep only active users
filtered_df = df[df['userId'].isin(active_users)]

print(f"Original number of ratings: {len(df)}")
print(f"Number of ratings after filtering users: {len(filtered_df)}")

Original number of ratings: 25000095
Number of ratings after filtering users: 23107403


In [10]:
# Count how many ratings each movie has received
movie_rating_counts = filtered_df['title'].value_counts()

# Identify movies that have 50 or more ratings
popular_movies = movie_rating_counts[movie_rating_counts >= 50].index

# Filter the dataframe to keep only popular movies
final_df = filtered_df[filtered_df['title'].isin(popular_movies)]

print(f"Number of ratings after filtering users: {len(filtered_df)}")
print(f"Number of ratings after filtering movies: {len(final_df)}")

Number of ratings after filtering users: 23107403
Number of ratings after filtering movies: 22754401


## Step 2: Creating the User-Item Interaction Matrix

We will now create a pivot table that represents our user-item interaction matrix. The rows will be `userId`, the columns will be movie `title`, and the values will be the `rating`. Missing values (where a user hasn't rated a movie) will be filled with 0.

In [ ]:
user_movie_matrix = final_df.pivot_table(index='userId', columns='title', values='rating').fillna(0)

print("Shape of the user-movie matrix:", user_movie_matrix.shape)
user_movie_matrix.head()

## Step 3: Decomposing the Matrix with SVD

Singular Value Decomposition (SVD) is a matrix factorization technique that reduces the dimensionality of our user-item matrix. It helps in discovering the latent (hidden) features that explain the ratings.

We will use `TruncatedSVD` because our matrix is sparse. We need to transpose the matrix so that movies are rows and users are columns, allowing SVD to find latent features for movies.

In [ ]:
# Transpose the matrix to have movies as rows and users as columns
X = user_movie_matrix.T

# Initialize TruncatedSVD
# n_components is the number of latent features we want to find (a hyperparameter)
svd = TruncatedSVD(n_components=12, random_state=17)

# Fit and transform the data
matrix_movie_features = svd.fit_transform(X)

print("Shape of the resulting movie-feature matrix:", matrix_movie_features.shape)

## Step 4: Making Recommendations with Cosine Similarity

Now that we have a matrix where each row represents a movie and the columns are its latent features, we can calculate the similarity between movies. We'll use **Cosine Similarity**, which measures the cosine of the angle between two vectors.

A similarity score of 1 means the movies are identical in terms of their latent features, and 0 means they are completely dissimilar.

In [ ]:
# Calculate the cosine similarity matrix
corr_matrix = cosine_similarity(matrix_movie_features)

# Create a Pandas DataFrame for the correlation matrix for better readability
corr_df = pd.DataFrame(corr_matrix, index=user_movie_matrix.columns, columns=user_movie_matrix.columns)

print("Shape of the correlation matrix:", corr_df.shape)
corr_df.head()

### Creating the Recommendation Function

Let's create a function that takes a movie title and returns a list of the top N most similar movies.

In [ ]:
def recommend_movies(movie_title, n_recommendations=10):
    """
    Recommends movies similar to a given movie title.
    
    Args:
        movie_title (str): The title of the movie to get recommendations for.
        n_recommendations (int): The number of recommendations to return.
        
    Returns:
        A list of recommended movie titles.
    """
    # Get the similarity scores for the given movie
    movie_correlations = corr_df[movie_title]
    
    # Sort the movies based on similarity scores in descending order
    similar_movies = movie_correlations.sort_values(ascending=False)
    
    # Exclude the movie itself (which will have a similarity of 1.0)
    recommended_movies = similar_movies.iloc[1:n_recommendations+1]
    
    print(f"Top {n_recommendations} recommendations for '{movie_title}':")
    for i, (title, score) in enumerate(recommended_movies.items()):
        print(f"{i+1}. {title} (Similarity: {score:.4f})")
    
    return recommended_movies.index.tolist()

## Step 5: Testing the Recommendation System

In [ ]:
# Test case 1: Get recommendations for "Forrest Gump (1994)"
recommend_movies("Forrest Gump (1994)")

In [ ]:
# Test case 2: Get recommendations for "Toy Story (1995)"
recommend_movies("Toy Story (1995)", n_recommendations=5)

In [ ]:
# Test case 3: Get recommendations for "Pulp Fiction (1994)"
recommend_movies("Pulp Fiction (1994)")

## Conclusion

We have successfully built a movie recommendation system using item-based collaborative filtering. The process involved:
1.  Loading and cleaning the MovieLens dataset.
2.  Filtering out inactive users and less popular movies to create a more robust dataset.
3.  Constructing a user-item interaction matrix.
4.  Using TruncatedSVD to perform dimensionality reduction and find latent movie features.
5.  Calculating cosine similarity to find the most similar movies.
6.  Creating a function to deliver the final recommendations.

The results show that the model is able to find movies that are thematically or contextually similar, which is the goal of a collaborative filtering system.